# SinLlama 1B — Inference
Loads the adapter from `isji/sinllama-1b-cpt` and runs Sinhala generation.

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers peft accelerate sentencepiece huggingface_hub hf_transfer

## 2. HuggingFace login

In [ ]:
from huggingface_hub import login
login(token="hf_your_token_here")

## 3. Load model + adapter

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

adapter_id = 'isji/sinllama-1b-cpt'
tokenizer_id = 'polyglots/Extended-Sinhala-LLaMA'

print('Loading extended Sinhala tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)

print('Loading base model ...')
config = PeftConfig.from_pretrained(adapter_id)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    dtype=torch.float16,
    device_map='auto',
)

# Extended Sinhala tokenizer has 139,336 tokens vs base 128,256
model.resize_token_embeddings(len(tokenizer))

print('Loading LoRA adapter ...')
model = PeftModel.from_pretrained(model, adapter_id)
model.eval()

print(f'Ready. Vocab size: {len(tokenizer):,} | Device: {model.device}')

## 4. Generate

In [ ]:
import torch
import warnings
from transformers import StoppingCriteria, StoppingCriteriaList

warnings.filterwarnings('ignore')

# --- Stop on ] token ---
class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids
    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

stop_tokens = ["]", "\n", "།"]
stop_token_ids = [
    tokenizer.encode(t, add_special_tokens=False)[0]
    for t in stop_tokens
    if tokenizer.encode(t, add_special_tokens=False)
]
if tokenizer.eos_token_id:
    stop_token_ids.append(tokenizer.eos_token_id)

stopping_criteria = StoppingCriteriaList([StopOnTokens(stop_token_ids)])

# --- Build bracket-format few-shot prompt ---
def build_prompt(context, question):
    prompt  = "උපදෙස්: පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් ප්‍රශ්නයට නිවැරදි, කෙටි පිළිතුරක් දෙන්න.\n\n"

    # Few-shot example 1
    prompt += "තොරතුරු (Context): ශ්‍රී ලංකාව ඉන්දියන් සාගරයේ පිහිටි දූපතකි. එහි නීතිමය අගනුවර ශ්‍රී ජයවර්ධනපුර කෝට්ටේ වේ.\n"
    prompt += "ප්‍රශ්නය: ශ්‍රී ලංකාවේ නීතිමය අගනුවර කුමක්ද?\n"
    prompt += "පිළිතුර: [ශ්‍රී ජයවර්ධනපුර කෝට්ටේ]\n\n"

    # Few-shot example 2
    prompt += "තොරතුරු (Context): ආදම්ගේ කන්ද (ශ්‍රී පාදය) යනු ශ්‍රී ලංකාවේ පිහිටි පූජනීය කන්දකි. එය මීටර් 2243 ක් උසය.\n"
    prompt += "ප්‍රශ්නය: ශ්‍රී පාදයේ උස කොපමණද?\n"
    prompt += "පිළිතුර: [මීටර් 2243]\n\n"

    # Few-shot example 3
    prompt += "තොරතුරු (Context): ජලය හයිඩ්‍රජන් හා ඔක්සිජන් වලින් සෑදී ඇත. එහි රසායනික සූත්‍රය H2O වේ.\n"
    prompt += "ප්‍රශ්නය: ජලයේ රසායනික සූත්‍රය කුමක්ද?\n"
    prompt += "පිළිතුර: [H2O]\n\n"

    # Actual question
    prompt += f"තොරතුරු (Context): {context}\n"
    prompt += f"ප්‍රශ්නය: {question}\n"
    prompt += "පිළිතුර: ["   # model fills inside the bracket
    return prompt

def generate(context, question):
    prompt = build_prompt(context, question)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,          # greedy — most reliable for factual answers
            repetition_penalty=1.15,
            stopping_criteria=stopping_criteria,
        )
    full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    raw = full_text.split("පිළිතුර: [")[-1]
    return raw.split("]")[0].strip()


# --- Tests ---
tests = [
    {
        "context": "විජය කුමරු ශ්‍රී ලංකාවේ පළමු රජු ලෙස සැලකේ. ඔහු තම්බපණ්ණි නගරය පිහිටුවීය. ඔහුට කුවේණි නම් බිරිඳක් සිටියාය.",
        "question": "ශ්‍රී ලංකාවේ පළමු රජු කවුද?",
    },
    {
        "context": "විජය කුමරු ශ්‍රී ලංකාවේ පළමු රජු ලෙස සැලකේ. ඔහු තම්බපණ්ණි නගරය පිහිටුවීය. ඔහුට කුවේණි නම් බිරිඳක් සිටියාය.",
        "question": "කුවේණි කියන්නේ කවුද?",
    },
    {
        "context": "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දූපත් රාජ්‍යකි. කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරයි. ශ්‍රී ජයවර්ධනපුර කෝට්ටේ නීතිමය අගනුවරයි.",
        "question": "ශ්‍රී ලංකාවේ වාණිජ අගනුවර කුමක්ද?",
    },
    {
        "context": "බෞද්ධ දර්ශනය අනුව, දුක්ඛ, සමුදය, නිරෝධ සහ මාර්ග යන චතුරාර්ය සත්‍ය හතරක් ඇත. මේවා ගෞතම බුදුරජාණන් වහන්සේ විසින් ප්‍රථමවරට දේශනා කරන ලදී.",
        "question": "චතුරාර්ය සත්‍ය කීයක් තිබේද?",
    },
    {
        "context": "සිංහල භාෂාව ශ්‍රී ලංකාවේ නිල භාෂාවකි. එය ඉන්දු-යුරෝපීය භාෂා පවුලට අයත් වේ.",
        "question": "සිංහල භාෂාව කුමන භාෂා පවුලට අයත්ද?",
    },
]

print("සිංහල AI — Few-Shot RAG\n" + "=" * 50)
for t in tests:
    answer = generate(t["context"], t["question"])
    print(f"Q: {t['question']}")
    print(f"A: {answer}")
    print("-" * 50) 

## 5. Interactive — type your own prompt

In [ ]:
prompt = "ඔබගේ ප්‍රශ්නය මෙහි ලියන්න"  # replace with your prompt
print(generate(prompt, max_new_tokens=200))